<a href="https://colab.research.google.com/github/skyexry/urban-mobility-forecast/blob/main/notebooks/02_graph_tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
import importlib.util
import sys
import numpy as np
from scipy.spatial.distance import cdist

In [7]:
!git clone https://github.com/skyexry/urban-mobility-forecast.git

Cloning into 'urban-mobility-forecast'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (15/15), done.
remote: Total 26 (delta 4), reused 21 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 6.70 KiB | 6.70 MiB/s, done.
Resolving deltas: 100% (4/4), done.


In [10]:
import sys
sys.path.append('/content/urban-mobility-forecast')

In [12]:
!git -C /content/urban-mobility-forecast pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 4 (delta 1), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 398 bytes | 398.00 KiB/s, done.
From https://github.com/skyexry/urban-mobility-forecast
   858a03c..2132766  main       -> origin/main
Updating 858a03c..2132766
Fast-forward
 preprocessing/__init__.py | 0
 1 file changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 preprocessing/__init__.py


In [19]:
spec = importlib.util.spec_from_file_location(
    "graph",
    "/content/urban-mobility-forecast/preprocessing/graph.py"
)
graph = importlib.util.module_from_spec(spec)
spec.loader.exec_module(graph)

print(dir(graph))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'adjacency_to_edge_index', 'build_adjacency_matrix', 'cdist', 'np', 'pd']


In [20]:
df = pd.read_parquet("/content/drive/MyDrive/citibike/hourly_demand_final.parquet")
W, stations = graph.build_adjacency_matrix(df)
edge_index, edge_weight = graph.adjacency_to_edge_index(W)
print(edge_index.shape, edge_weight.shape)

Stations     : 438
Non-zero edges: 191406
Sparsity     : 0.23%
(2, 191406) (191406,)


In [26]:
# Compute pairwise distances to determine sigma2
coords = stations[['start_lat', 'start_lng']].values
dist_matrix = cdist(coords, coords, metric='euclidean')
distances = dist_matrix[dist_matrix > 0]

print(f"Distance stats — min: {distances.min():.4f}, mean: {distances.mean():.4f}, max: {distances.max():.4f}")

# Grid search over sigma2 and theta
for sigma2 in [0.001, 0.002, 0.005]:
    for theta in [0.3, 0.5, 0.7]:
        W_test, _ = graph.build_adjacency_matrix(df, sigma2=sigma2, theta=theta)
        edges = np.count_nonzero(W_test)
        sparsity = 1 - edges / (len(stations) ** 2)
        print(f"sigma2={sigma2}, theta={theta}: {edges} edges, sparsity={sparsity:.2%}")

Distance stats — min: 0.0007, mean: 0.0427, max: 0.1553
Stations     : 438
Non-zero edges: 81932
Sparsity     : 57.29%
sigma2=0.001, theta=0.3: 81932 edges, sparsity=57.29%
Stations     : 438
Non-zero edges: 55988
Sparsity     : 70.82%
sigma2=0.001, theta=0.5: 55988 edges, sparsity=70.82%
Stations     : 438
Non-zero edges: 33324
Sparsity     : 82.63%
sigma2=0.001, theta=0.7: 33324 edges, sparsity=82.63%
Stations     : 438
Non-zero edges: 122392
Sparsity     : 36.20%
sigma2=0.002, theta=0.3: 122392 edges, sparsity=36.20%
Stations     : 438
Non-zero edges: 89378
Sparsity     : 53.41%
sigma2=0.002, theta=0.5: 89378 edges, sparsity=53.41%
Stations     : 438
Non-zero edges: 57232
Sparsity     : 70.17%
sigma2=0.002, theta=0.7: 57232 edges, sparsity=70.17%
Stations     : 438
Non-zero edges: 173294
Sparsity     : 9.67%
sigma2=0.005, theta=0.3: 173294 edges, sparsity=9.67%
Stations     : 438
Non-zero edges: 146076
Sparsity     : 23.86%
sigma2=0.005, theta=0.5: 146076 edges, sparsity=23.86%
Stat